# UPI Transaction Data Analysis
**Pondicherry University — School of Management, Department of Banking Technology**  
**MBAF426 — Business Intelligence Project**

**Team Members:** Harine Navanitha A.P | Harini V | Udhayasri S  
**Guide:** Dr. C. Punitha Devi

---

## Objective
Analyse 5 years (FY2019-20 to FY2023-24) of NPCI UPI transaction data to derive actionable insights on:
- Transaction volume and value trends
- Banking ecosystem growth
- P2P vs P2M transaction behaviour
- YoY and MoM growth patterns

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE = '#1F4E79'
ORANGE = '#ED7D31'
GREEN = '#70AD47'
RED = '#C00000'
LIGHT_BLUE = '#2E75B6'

print('Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/UPI_Transaction_Data.csv')
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b-%Y')
df['FY'] = df['Month_dt'].apply(lambda x: f"FY{x.year-1}-{str(x.year)[2:]}" if x.month < 4 else f"FY{x.year}-{str(x.year+1)[2:]}")

print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

## 3. Data Overview & Summary Statistics

In [ ]:
print('=== Dataset Info ===')
print(f'Months covered : {df["Month"].iloc[0]} to {df["Month"].iloc[-1]}')
print(f'Financial Years: {df["FY"].nunique()} FYs')
print(f'Missing values : {df.isnull().sum().sum()} (only Volume_Growth and Value_Growth for first month)')
print()

summary = df[['Total_Volume_Crore','Total_Value_Lakh_Crore','Avg_Txn_Value_INR','Banks_Live_on_UPI']].describe().round(2)
print('=== Summary Statistics ===')
print(summary)

## 4. Transaction Volume Trend

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Volume
ax1 = axes[0]
ax1.fill_between(range(len(df)), df['Total_Volume_Crore'], alpha=0.15, color=BLUE)
ax1.plot(range(len(df)), df['Total_Volume_Crore'], color=BLUE, linewidth=2.5, marker='o', markersize=4)
ax1.set_title('UPI Monthly Transaction Volume (Crore)', fontsize=13, fontweight='bold', color=BLUE)
ax1.set_ylabel('Volume (Crore)', fontsize=10)
ax1.set_xticks(range(0, len(df), 6))
ax1.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 6)], rotation=45, ha='right', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Add annotation for key milestone
peak_idx = df['Total_Volume_Crore'].idxmax()
ax1.annotate(f'Peak: {df["Total_Volume_Crore"].max():,.0f} Cr\n{df["Month"].iloc[peak_idx]}',
             xy=(peak_idx, df['Total_Volume_Crore'].max()),
             xytext=(peak_idx-8, df['Total_Volume_Crore'].max()*0.88),
             arrowprops=dict(arrowstyle='->', color=RED),
             fontsize=9, color=RED, fontweight='bold')

# Value
ax2 = axes[1]
ax2.fill_between(range(len(df)), df['Total_Value_Lakh_Crore'], alpha=0.15, color=ORANGE)
ax2.plot(range(len(df)), df['Total_Value_Lakh_Crore'], color=ORANGE, linewidth=2.5, marker='o', markersize=4)
ax2.set_title('UPI Monthly Transaction Value (Lakh Crore INR)', fontsize=13, fontweight='bold', color=ORANGE)
ax2.set_ylabel('Value (Lakh Crore)', fontsize=10)
ax2.set_xticks(range(0, len(df), 6))
ax2.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 6)], rotation=45, ha='right', fontsize=9)

plt.tight_layout(pad=2)
plt.savefig('../dashboard/Transaction_Trends.png', bbox_inches='tight')
plt.show()
print(f"Volume grew from {df['Total_Volume_Crore'].iloc[0]:,} Cr (Apr-2019) to {df['Total_Volume_Crore'].iloc[-1]:,} Cr (Mar-2024)")
print(f"That is a {((df['Total_Volume_Crore'].iloc[-1]/df['Total_Volume_Crore'].iloc[0])-1)*100:.1f}x growth in 5 years")

## 5. YoY Growth Analysis by Financial Year

In [ ]:
fy_summary = df.groupby('FY').agg(
    Annual_Volume=('Total_Volume_Crore','sum'),
    Annual_Value=('Total_Value_Lakh_Crore','sum'),
    Avg_Txn_Value=('Avg_Txn_Value_INR','mean'),
    Avg_Banks=('Banks_Live_on_UPI','mean')
).round(2).reset_index()

fy_summary['Vol_YoY_Pct'] = fy_summary['Annual_Volume'].pct_change().mul(100).round(1)
fy_summary['Val_YoY_Pct'] = fy_summary['Annual_Value'].pct_change().mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual Volume bars
bars = axes[0].bar(fy_summary['FY'], fy_summary['Annual_Volume'], color=BLUE, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].set_title('Annual UPI Volume by Financial Year', fontsize=12, fontweight='bold', color=BLUE)
axes[0].set_ylabel('Total Volume (Crore)', fontsize=10)
axes[0].set_xticklabels(fy_summary['FY'], rotation=30, ha='right')
for bar, val in zip(bars, fy_summary['Annual_Volume']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01, f'{val:,.0f}', ha='center', fontsize=8, fontweight='bold')

# YoY Growth rate
yoy = fy_summary.dropna(subset=['Vol_YoY_Pct'])
colors = [GREEN if x > 0 else RED for x in yoy['Vol_YoY_Pct']]
axes[1].bar(yoy['FY'], yoy['Vol_YoY_Pct'], color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('YoY Volume Growth (%) by Financial Year', fontsize=12, fontweight='bold', color=GREEN)
axes[1].set_ylabel('YoY Growth (%)', fontsize=10)
axes[1].set_xticklabels(yoy['FY'], rotation=30, ha='right')
axes[1].axhline(0, color='black', linewidth=0.8)
for bar, val in zip(axes[1].patches, yoy['Vol_YoY_Pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02, f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../dashboard/Growth_Analysis.png', bbox_inches='tight')
plt.show()
print(fy_summary[['FY','Annual_Volume','Annual_Value','Vol_YoY_Pct','Val_YoY_Pct']].to_string(index=False))

## 6. Banking Ecosystem — Banks Live on UPI

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(df)), df['Banks_Live_on_UPI'], color=GREEN, linewidth=2.5, marker='s', markersize=3)
ax.fill_between(range(len(df)), df['Banks_Live_on_UPI'], alpha=0.15, color=GREEN)
ax.set_title('Number of Banks Live on UPI (Apr 2019 – Mar 2024)', fontsize=13, fontweight='bold', color=GREEN)
ax.set_ylabel('Number of Banks', fontsize=10)
ax.set_xticks(range(0, len(df), 6))
ax.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 6)], rotation=45, ha='right', fontsize=9)
ax.annotate(f"Apr 2019: {df['Banks_Live_on_UPI'].iloc[0]} banks", xy=(0, df['Banks_Live_on_UPI'].iloc[0]),
            xytext=(4, df['Banks_Live_on_UPI'].iloc[0]-20), fontsize=9, color=GREEN, fontweight='bold')
ax.annotate(f"Mar 2024: {df['Banks_Live_on_UPI'].iloc[-1]} banks", xy=(len(df)-1, df['Banks_Live_on_UPI'].iloc[-1]),
            xytext=(len(df)-15, df['Banks_Live_on_UPI'].iloc[-1]-25), fontsize=9, color=GREEN, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Banks grew from {df['Banks_Live_on_UPI'].iloc[0]} (Apr 2019) to {df['Banks_Live_on_UPI'].iloc[-1]} (Mar 2024) — +{df['Banks_Live_on_UPI'].iloc[-1]-df['Banks_Live_on_UPI'].iloc[0]} banks in 5 years")

## 7. P2P vs P2M Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked area chart — volume
axes[0].stackplot(range(len(df)), df['P2P_Volume_Crore'], df['P2M_Volume_Crore'],
                  labels=['P2P (Person-to-Person)', 'P2M (Person-to-Merchant)'],
                  colors=[BLUE, ORANGE], alpha=0.75)
axes[0].set_title('P2P vs P2M Volume Trend (Crore)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Volume (Crore)', fontsize=10)
axes[0].set_xticks(range(0, len(df), 12))
axes[0].set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 12)], rotation=30, ha='right', fontsize=9)
axes[0].legend(loc='upper left', fontsize=9)

# P2M share over time
df['P2M_Share_Pct'] = df['P2M_Volume_Crore'] / df['Total_Volume_Crore'] * 100
axes[1].plot(range(len(df)), df['P2M_Share_Pct'], color=ORANGE, linewidth=2.5)
axes[1].fill_between(range(len(df)), df['P2M_Share_Pct'], 30, alpha=0.15, color=ORANGE)
axes[1].set_title('P2M Share of Total UPI Volume (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('P2M Share (%)', fontsize=10)
axes[1].set_xticks(range(0, len(df), 12))
axes[1].set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 12)], rotation=30, ha='right', fontsize=9)
axes[1].set_ylim(25, 60)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1, label='50% mark')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f"P2M share grew from {df['P2M_Share_Pct'].iloc[0]:.1f}% (Apr 2019) to {df['P2M_Share_Pct'].iloc[-1]:.1f}% (Mar 2024)")
print("Insight: Merchant payments now account for over half of all UPI transactions — driven by QR code adoption")

## 8. Average Transaction Value Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(df)), df['Avg_Txn_Value_INR'], color=RED, linewidth=2.5, marker='D', markersize=3)
ax.axhline(df['Avg_Txn_Value_INR'].mean(), color='gray', linestyle='--', linewidth=1, label=f"5-yr Avg: ₹{df['Avg_Txn_Value_INR'].mean():.0f}")
ax.set_title('Average UPI Transaction Value (INR)', fontsize=13, fontweight='bold', color=RED)
ax.set_ylabel('Avg Txn Value (INR)', fontsize=10)
ax.set_xticks(range(0, len(df), 6))
ax.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 6)], rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()
print(f"Avg txn value: ₹{df['Avg_Txn_Value_INR'].iloc[0]:.0f} (Apr 2019) → ₹{df['Avg_Txn_Value_INR'].iloc[-1]:.0f} (Mar 2024)")
print("Insight: Rising avg value signals shift from small everyday payments to higher-value transactions")

## 9. Executive Overview Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#F0F4F8')

# Title
fig.text(0.5, 0.97, 'UPI Transaction Data Analysis — Executive Dashboard',
         ha='center', fontsize=16, fontweight='bold', color=BLUE)
fig.text(0.5, 0.94, 'FY 2019-20 to FY 2023-24 | Source: NPCI Monthly Statistics',
         ha='center', fontsize=10, color='gray')

# KPI boxes
kpis = [
    (f"{df['Total_Volume_Crore'].sum():,.0f} Cr", "Total 5-Yr Volume", BLUE),
    (f"₹{df['Total_Value_Lakh_Crore'].sum():,.0f} L Cr", "Total 5-Yr Value", ORANGE),
    (f"22.3x", "Volume Growth (5yr)", GREEN),
    (f"{df['Banks_Live_on_UPI'].iloc[-1]}", "Banks on UPI (Mar'24)", RED),
]

for i, (val, label, color) in enumerate(kpis):
    ax = fig.add_axes([0.05 + i*0.235, 0.79, 0.21, 0.12])
    ax.set_facecolor(color)
    ax.text(0.5, 0.6, val, ha='center', va='center', transform=ax.transAxes, fontsize=16, fontweight='bold', color='white')
    ax.text(0.5, 0.15, label, ha='center', va='center', transform=ax.transAxes, fontsize=9, color='white')
    ax.set_xticks([]); ax.set_yticks([])

# Volume trend
ax1 = fig.add_axes([0.05, 0.42, 0.55, 0.33])
ax1.fill_between(range(len(df)), df['Total_Volume_Crore'], alpha=0.2, color=BLUE)
ax1.plot(range(len(df)), df['Total_Volume_Crore'], color=BLUE, linewidth=2)
ax1.set_title('Monthly Volume Trend (Crore Transactions)', fontsize=10, fontweight='bold', color=BLUE)
ax1.set_xticks(range(0, len(df), 12))
ax1.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 12)], fontsize=8, rotation=20)
ax1.set_facecolor('#F8FBFF')

# P2M share
ax2 = fig.add_axes([0.64, 0.42, 0.32, 0.33])
ax2.stackplot(range(len(df)), df['P2P_Volume_Crore'], df['P2M_Volume_Crore'],
              labels=['P2P','P2M'], colors=[BLUE, ORANGE], alpha=0.7)
ax2.set_title('P2P vs P2M Volume', fontsize=10, fontweight='bold')
ax2.set_xticks(range(0, len(df), 12))
ax2.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 12)], fontsize=8, rotation=20)
ax2.legend(fontsize=8, loc='upper left')
ax2.set_facecolor('#F8FBFF')

# Annual volume bar
ax3 = fig.add_axes([0.05, 0.06, 0.40, 0.30])
bars = ax3.bar(fy_summary['FY'], fy_summary['Annual_Volume'], color=BLUE, alpha=0.8, edgecolor='white')
ax3.set_title('Annual Volume by FY (Crore)', fontsize=10, fontweight='bold', color=BLUE)
ax3.set_xticklabels(fy_summary['FY'], rotation=20, fontsize=8)
ax3.set_facecolor('#F8FBFF')

# Avg txn value
ax4 = fig.add_axes([0.52, 0.06, 0.44, 0.30])
ax4.plot(range(len(df)), df['Avg_Txn_Value_INR'], color=RED, linewidth=2)
ax4.fill_between(range(len(df)), df['Avg_Txn_Value_INR'], alpha=0.15, color=RED)
ax4.set_title('Avg Transaction Value (INR)', fontsize=10, fontweight='bold', color=RED)
ax4.set_xticks(range(0, len(df), 12))
ax4.set_xticklabels([df['Month'].iloc[i] for i in range(0, len(df), 12)], fontsize=8, rotation=20)
ax4.set_facecolor('#F8FBFF')

plt.savefig('../dashboard/Executive_Overview.png', bbox_inches='tight', dpi=150)
plt.show()
print('Executive dashboard saved to ../dashboard/Executive_Overview.png')

## 10. Key Findings & Business Insights

In [ ]:
print('=' * 65)
print('KEY FINDINGS — UPI Transaction Data Analysis')
print('=' * 65)

print(f"""
1. EXPLOSIVE GROWTH
   Volume : {df['Total_Volume_Crore'].iloc[0]:,} Cr (Apr-2019) → {df['Total_Volume_Crore'].iloc[-1]:,} Cr (Mar-2024)
   Value  : ₹{df['Total_Value_Lakh_Crore'].iloc[0]:.1f} L Cr → ₹{df['Total_Value_Lakh_Crore'].iloc[-1]:.1f} L Cr
   UPI transaction volume grew over 22x in just 5 years.

2. BANKING ECOSYSTEM EXPANSION
   Banks on UPI grew from {df['Banks_Live_on_UPI'].iloc[0]} to {df['Banks_Live_on_UPI'].iloc[-1]}, reflecting broad
   adoption across public, private, co-operative and RRBs.

3. P2M RISING — MERCHANT ADOPTION
   P2M share rose from ~30% (FY20) to ~55% (FY24).
   Signals QR code proliferation and merchant digitisation.

4. AVERAGE TRANSACTION VALUE INCREASING
   ₹{df['Avg_Txn_Value_INR'].iloc[0]:.0f} → ₹{df['Avg_Txn_Value_INR'].iloc[-1]:.0f} per transaction.
   Users are moving beyond micro-payments to utilities, insurance, EMIs.

5. PANDEMIC ACCELERATION (FY21 → FY22)
   COVID-19 drove contactless payment adoption, evident in the
   volume surge from FY21 to FY22.

6. FINTECH POLICY IMPACT
   RBI and NPCI policies (UPI Lite, credit on UPI, interoperability)
   have continuously expanded the ecosystem.
""")
print('=' * 65)